# Week 7 capstone: Step 4, step-budget calibration (stages 1 and 2)

**Scope: calibration only.** This notebook runs the calibration runs that the pre-registered training-loss rule asks for, and nothing else. There is no noise-pilot cell and no main-experiment cell. The repository also refuses those stages until their own gates are met.

- **What runs.** Arm A (learned absolute), seed 900, at step counts 1000, 2000 and 4000 (stage 1). Then arms B and C at the chosen step count and the next one (stage 2), moving up if an arm has not plateaued. About 7 runs, roughly 15-20 minutes on a T4. The driver decides what to run and stops by itself.
- **The step budget is chosen from training loss only.** Arms B and C do not compute any validation or test metric in this stage, so no between-arm validation number exists to look at.
- **Policy P1 (fully deterministic) is used for every run.** It was selected in two smoke sessions and approved.
- **No `pip install`, no upgrades.** The environment is checked against `requirements.lock.txt`, and the run is refused if it differs.

Run the cells top to bottom on a **T4 GPU runtime**, then download the zip from the last cell and send it back. The step count is set only after the results are reviewed.

In [ ]:
# 1. Get the code. Two ways, both leave a git checkout in ./repo:
#    (a) upload week7_repo.zip to the Colab Files panel (left sidebar) BEFORE running this cell; or
#    (b) leave it out and this cell clones from GitHub (edit REPO_URL if yours differs).
#    Optionally pin COMMIT to an exact commit hash. The exact hash is printed and recorded either way.
REPO_URL = "https://github.com/sweiss1367/wk5_architecture-variants.git"
BRANCH = "claude/trusting-pascal-1qjsp0"
COMMIT = ""   # paste the commit hash given with the zip, or leave empty for the branch head

import os, subprocess, time
if os.path.isdir("repo"):                  # never reuse a stale copy: keep the old one aside
    os.rename("repo", f"repo_old_{int(time.time())}")
if os.path.exists("week7_repo.zip"):
    subprocess.run(["unzip", "-q", "week7_repo.zip"], check=True)      # creates ./repo, including its .git history
else:
    url = REPO_URL
    try:                                   # private repo: add a Colab secret named GITHUB_TOKEN (never printed)
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN")
        if tok:
            url = REPO_URL.replace("https://", f"https://{tok}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "--branch", BRANCH, url, "repo"], check=True)
    subprocess.run(["git", "-C", "repo", "remote", "set-url", "origin", REPO_URL], check=True)   # drop the token from the remote
if COMMIT:
    subprocess.run(["git", "-C", "repo", "checkout", COMMIT], check=True)
%cd repo/week7_capstone
!git rev-parse HEAD
!git status --short

/content/repo/week7_capstone
5d9843a634ecba0d48308f960972b62a2e56e181


In [ ]:
# 2. GPU check (the smoke sessions used a Tesla T4; a different GPU is allowed but changes absolute timings).
!nvidia-smi --query-gpu=name,uuid,driver_version --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: switch the runtime to a T4 GPU and restart."
name = torch.cuda.get_device_name(0)
print(torch.__version__, torch.version.cuda, name)
if "T4" not in name:
    print("WARNING: not a T4. Calibration will still run, but tell the reviewer.")

name, uuid, driver_version
Tesla T4, GPU-359219fc-e208-ac27-b4fa-a28db8756af3, 580.82.07
2.11.0+cu130 13.0 Tesla T4


In [ ]:
# 3. Gates, checked before any run: environment equals the lock, code equals the commit, and the recorded policy decision
#    agrees with the config. (run_ablation.py checks all of these again for every run.)
import json, subprocess, yaml
subprocess.run(["python", "src/check_env.py", "--verify"], check=True)
subprocess.run(["git", "diff", "--quiet"], check=True)                    # no tracked file differs from the commit
decision = json.load(open("results/determinism_decision.json"))["selected_policy"]
config = yaml.safe_load(open("configs/base.yaml"))["determinism"]["policy"]
print("policy decision:", decision, "| config policy:", config)
assert decision == config == "P1"

policy decision: P1 | config policy: P1


In [ ]:
# 4. Calibration. The driver runs only what the rule needs, one fresh process per run, and then prints the decision table.
#    A failed run stops it with the traceback. Re-running this cell resumes: finished runs are skipped.
!python src/run_calibration.py

A seed 900 [calibration] steps=1000 val_mape=5.438 train_s=35.7 final_train_loss=0.0533 -> results/runs/calibration/A_seed900_S1000_attempt1.json
A seed 900 [calibration] steps=2000 val_mape=4.974 train_s=69.8 final_train_loss=0.0484 -> results/runs/calibration/A_seed900_S2000_attempt1.json
A seed 900 [calibration] steps=4000 val_mape=4.748 train_s=143.7 final_train_loss=0.0453 -> results/runs/calibration/A_seed900_S4000_attempt1.json
-- 3 runs so far; checking whether the rule needs more --
-- no further runs needed (3 run this invocation) --

arm   steps  final train loss   train s   steps/s
A      1000           0.05330      35.7      28.0
A      2000           0.04839      69.8      28.6
A      4000           0.04531     143.7      27.8

stage 1 (arm A):
  S=1000 -> 2000: improvement 9.22%  qualifies=False
  S=2000 -> 4000: improvement 6.36%  qualifies=False

STATUS: no_grid_value_qualifies | recommended S: None | validation used: False
NEXT_ACTION: expansion trigger: add S=8000 fo

In [ ]:
# 5. Package the calibration outputs (kept apart from smoke and main results) and download them.
import glob, os, subprocess
paths = ["results/runs/calibration", "results/calibration_decision.json", "results/env_report.json",
         "results/session_manifest.json"] + glob.glob("results/pip_freeze_*.txt")
paths = [p for p in paths if os.path.exists(p)]
subprocess.run(["zip", "-rq", "week7_calibration_outputs.zip", *paths], check=True)
print("zipped:", paths)
from google.colab import files
files.download("week7_calibration_outputs.zip")

zipped: ['results/runs/calibration', 'results/calibration_decision.json', 'results/env_report.json', 'results/session_manifest.json', 'results/pip_freeze_b506fea2eb.txt']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Stop here.** Send back `week7_calibration_outputs.zip`. The next steps (set `train.steps`, then the 2-run noise pilot on arm A, then the pre-registration freeze) happen only after the results are reviewed.